# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**Content-level distributions, same grain ML-07 scores on.** Heavy right tail on impressions
(a handful of pages carry a huge share of traffic — the top page in the baseline was ~7% of
its entire position bucket's impression volume) and on position (most content sits past
position 10, only a small slice is in the top 3). Both matter: a mean CTR across all pages
would be dominated by a few outliers, which is exactly why the baseline uses a position-bucket
average instead of a single global average.


In [2]:
import pandas as pd

DATA_DIR = "../../"

perf = pd.read_parquet(
    f"{DATA_DIR}/fact_content_daily_performance_sample.parquet",
    columns=[
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
)

agg = perf.groupby(
    ["client_hash_id", "content_hash_id"]
).agg(
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum"),
    avg_pos=("gsc_avg_position", "mean")
).reset_index()

agg = agg[agg.impressions >= 50].copy()

agg["ctr"] = agg.clicks / agg.impressions

print(agg[["impressions", "clicks", "avg_pos", "ctr"]].describe())

         impressions         clicks        avg_pos            ctr
count  120681.000000  120681.000000  120681.000000  120681.000000
mean     1781.896164       9.891831      20.214757       0.004107
std      6644.776837     602.059047      18.218145       0.006690
min        50.000000       0.000000       0.000000       0.000000
25%       146.000000       0.000000       7.343871       0.000000
50%       386.000000       1.000000      12.917363       0.002132
75%      1233.000000       5.000000      27.112644       0.005803
max    615012.000000  152170.000000     184.274194       0.536745


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Signal #1 — CTR vs. position (flag-linked: FlyRank's "Fix CTR" optimization flag).**
Verdict: **CONFIRMED**. From the executed run in ML-07:

| pos_bucket | n | ctr |
|---|---|---|
| 1-3 | 2,199 | 4.41% |
| 3-5 | 8,894 | 0.68% |
| 5-10 | 37,261 | 0.35% |
| 10-20 | 30,258 | 0.41% |
| 20-50 | 31,204 | 0.26% |
| 50+ | 10,863 | 0.06% |

Position 1-3 converts 6-13x higher than every other bucket. There's a small non-monotonic wobble
(10-20 sits slightly above 5-10) — with 30k+ rows per bucket that's noise around a flat middle,
not a real reversal. The core pattern holds strongly.

**Signal #2 — staleness (days since `content_updated_date`) vs. within-month click decline.**
Verdict: **FALSE**. From the same executed run:

| staleness_bucket | n | pct_declined |
|---|---|---|
| ≤30d | 26,736 | 54.0% |
| 30-90d | 25,884 | 52.7% |
| 90-180d | 8,293 | 53.4% |
| 180-365d | 84 | 52.4% |

`pct_declined` is flat — 52-54% regardless of how stale the content is. If staleness drove
decline, this should climb with age; it doesn't move. Negative result, kept as-is (not forced
into the baseline rule — see ML-07).


In [5]:
dc = pd.read_parquet(
    f"{DATA_DIR}/dim_content.parquet",
    columns=[
        "client_hash_id",
        "content_hash_id",
        "content_updated_date"
    ]
)

# Make sure the date column is actually datetime
dc["content_updated_date"] = pd.to_datetime(
    dc["content_updated_date"],
    errors="coerce"
)

# Make sure performance dates are datetime too
perf["report_date"] = pd.to_datetime(
    perf["report_date"],
    errors="coerce"
)

month_end = perf["report_date"].max()

staleness = agg.merge(
    dc,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Calculate age of content in days
staleness["days_stale"] = (
    month_end - staleness["content_updated_date"]
).dt.days

bins_stale = [-1, 30, 90, 180, 365, 999999]
stale_labels = ["<=30d", "30-90d", "90-180d", "180-365d", "365d+"]

staleness["staleness_bucket"] = pd.cut(
    staleness["days_stale"],
    bins=bins_stale,
    labels=stale_labels
).astype(str)

print(staleness[[
    "content_hash_id",
    "content_updated_date",
    "days_stale",
    "staleness_bucket"
]].head())

            content_hash_id content_updated_date  days_stale staleness_bucket
0  content_0058bd88fb1821f2           2026-06-13          17            <=30d
1  content_0059a4d4195810c9           2026-06-23           7            <=30d
2  content_005b6b7f7b8dda7f           2026-06-23           7            <=30d
3  content_0094c7d0fbcc07b7           2026-06-01          29            <=30d
4  content_00a34394d4ee05ce           2026-06-15          15            <=30d


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**FlyRank's "Fix CTR" optimization flag assumes pages ranking well but converting poorly deserve
a snippet/metadata rewrite — i.e., it assumes position and CTR are related enough that a gap
between them is actionable.** Signal #1 above directly tests that assumption and it holds
(CONFIRMED). It's also independently backed outside this dataset: the published FlyRank
research paper (`docs/flyrank-seo-research-march-2026.pdf`, Finding #3, "Click Capture by
Position Tier") finds the same directional pattern at full-portfolio scale — weighted CTR drops
88% from Top-3 (0.423%) to Deep (0.050%) positions. Two independent measurements, same
direction — this is the signal I'd trust most for the baseline rule.

By contrast, **no FlyRank flag I'm aware of relies on staleness alone** — and Signal #2 is
exactly why that's reasonable: staleness alone doesn't predict decline in this data.


In [6]:
# No new computation needed — this section interprets Signal 1's result against the
# "Fix CTR" flag's implicit assumption and the external paper finding, both already measured above.
print("Fix CTR flag assumption: position predicts CTR gap -> supported by Signal 1 (CONFIRMED)")
print("Cross-check: paper Finding #3 shows the same direction at portfolio scale (88% CTR drop, Top-3 -> Deep)")


Fix CTR flag assumption: position predicts CTR gap -> supported by Signal 1 (CONFIRMED)
Cross-check: paper Finding #3 shows the same direction at portfolio scale (88% CTR drop, Top-3 -> Deep)


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team should trust **CTR-gap-vs-position** as a real triage signal — it's confirmed
twice, once in this warehouse slice and once independently in FlyRank's own published portfolio
study. They should **not** treat "this page hasn't been updated in a while" as a reason on its
own to prioritize it — our data shows staleness alone doesn't predict which pages are actually
declining, so a staleness-only rule would waste review time on pages that are simply old but
fine, and could miss ones that are genuinely underperforming despite being recently updated.


In [7]:
print("Practice takeaway: prioritize by CTR gap vs. position, not by content age alone.")

Practice takeaway: prioritize by CTR gap vs. position, not by content age alone.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.